#Requirements

In [1]:
import cv2 as cv
import numpy as np
from matplotlib import pyplot as plt
import json
from skimage.exposure import match_histograms
import imagehash
from shapely.geometry import Polygon
from PIL import Image
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

/workspace16/Shayan/Shayan-env/lib64/python3.6/site-packages/google/auth/crypt/_cryptography_rsa.py:22: CryptographyDeprecationWarning: Python 3.6 is no longer supported by the Python core team. Therefore, support for it is deprecated in cryptography. The next release of cryptography (40.0) will be the last to support Python 3.6.
  import cryptography.exceptions


# Dataset Pre-processing

In [2]:
def apply_threshold(img_gray):
  ret, thresh = cv.threshold(img_gray, 250.00, 255.00, int(cv.THRESH_OTSU))
  # plt.imshow(cv.cvtColor(thresh, cv.COLOR_BGR2RGB))
  # plt.title('After applying OTSU threshold')
  # plt.show()
  return thresh

In [3]:
def hist_equ(image):

    size = image.shape[0:2]
    image_size = size[0]*size[1]
    L = 256

    unique, counts = np.unique(image, return_counts=True)
    pxl_count = dict(zip(unique, counts))

    key = [*pxl_count]
    key.sort() # we can remove this line
    list_sum = [] # sigma (𝑛𝑗 / 𝑛)
    counter = 0

    new_pxl = {}
    
    for item in key:
        counter += pxl_count[item]
        new_pxl.update({item:int((counter/image_size) * (L-1))})


    # print(len(image[0]))
    for row in range(size[0]):
        for col in range(size[1]):
            image[row][col] = new_pxl[image[row][col]]

    return image

In [4]:
def morphology(img_binary, kernel = np.ones((35, 35), np.uint8)):
  opening = cv.morphologyEx(img_binary, cv.MORPH_OPEN, kernel)
  closing = cv.morphologyEx(opening, cv.MORPH_CLOSE, kernel)
  # plt.imshow(cv.cvtColor(closing, cv.COLOR_BGR2RGB))
  # plt.title('After applying morphology transformation')
  # plt.show()
  return closing

In [5]:
def crop_image(img, img_binary):
  temp = np.where(img_binary==255)
  crop_indexes = min(temp[0]) + 5, min(temp[1]) + 5
  mask = img_binary==255
  img_cropped = img[np.ix_(mask.any(1), mask.any(0))]
  img_cropped = img_cropped[5:-5, 5:-5]
  # plt.imshow(cv.cvtColor(img_cropped, cv.COLOR_BGR2RGB))
  # plt.title('After cropping_image')
  # plt.show()
  return img_cropped, crop_indexes

In [6]:
def histogram_matching(img_cropped, pattern):
  m = match_histograms(img_cropped, pattern)
  # plt.imshow(cv.cvtColor(m, cv.COLOR_BGR2RGB))
  # plt.title('After histogram matching')
  # plt.show()
  return m

In [7]:
def rotate_finder(img_matched,pattern_resized):
    simil = []
    
    im_pil1 = Image.fromarray(img_matched)
    hash = imagehash.average_hash(im_pil1)
    rotate = [None,cv.ROTATE_90_CLOCKWISE,cv.ROTATE_180,cv.ROTATE_90_COUNTERCLOCKWISE]

    
    for angle in rotate:
        if angle != None:
            img = cv.rotate(pattern_resized, angle)
            im_pil2 = Image.fromarray(img)
            hash_ = imagehash.average_hash(im_pil2)
            simil.append(hash - hash_)
        else:
            im_pil2 = Image.fromarray(pattern_resized)
            hash_ = imagehash.average_hash(im_pil2)
            simil.append(hash - hash_)
    indx = rotate[simil.index(min(simil))]
    hold = cv.rotate(pattern_resized, indx)
    
    if hold.shape[0] != img_matched.shape[0]:
        hold = cv.resize(hold, (hold.shape[1], hold.shape[0]))
    
    return hold
    

In [8]:
def extract_pattern(img_name):
    json_name = img_name[:-3] + "json"
    f = open(json_name, encoding="utf8")
    data = json.load(f)
    f.close()
    pattern = cv.imread('Patterns/' + data['pattern'])
    return pattern

In [9]:
def find_label(img_name,i,window_size_height, j,window_size_width,main):
    not_main = Polygon([(i,j),(i+window_size_height,j),(i+window_size_height,j+window_size_width),(i,j+window_size_width)]).buffer(0)
    Flag = 0
    inter = not_main.intersection(main).area
    if inter > 0:
        return 1
    return 0

In [10]:
def window_sliding(img, pattern, img_name, crop_indexes, window_size_height=120, window_size_width=120):
  images, patterns, labels = [], [], []
  height, width= img.shape
  point_list = []
  json_name = img_name[:-3] + "json"
  f = open(json_name, encoding="utf8")
  data = json.load(f)
  f.close()
  for shape in data["shapes"]:
    points = np.array(shape['points']) - crop_indexes
    for pxl in points:
        point_list.append((int(pxl[0]),int(pxl[1])))
  main = Polygon(point_list).buffer(0)
        
  for i in range(0, height-window_size_height, window_size_height):
    for j in range(0, width-window_size_width, window_size_width):
      # cv.imwrite(dir +'/' + str(i)+str(j)+'_img_'+img_name, img_part)
      # cv.imwrite(dir +'/' + str(i)+str(j)+'_pattern_'+img_name, pattern_part)
      temp = pattern[i:i+window_size_height, j:j+window_size_width]
      if temp.shape==(120,120):
        patterns.append(temp)
        images.append(img[i:i+window_size_height, j:j+window_size_width])
        labels.append(find_label(img_name,i,window_size_height, j,window_size_width,main))
  return images, patterns, labels

In [11]:
def pre_processing(img_name,Kernel_size = 120):
  img = cv.imread(img_name).astype('uint8')
  img_gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
  img_binary = apply_threshold(img_gray)
  img_binary = morphology(img_binary)
  img_cropped, crop_indexes = crop_image(img, img_binary)
  pattern = extract_pattern(img_name)
  pattern = cv.cvtColor(pattern, cv.COLOR_BGR2GRAY)
  img_matched = histogram_matching(cv.equalizeHist(cv.cvtColor(img_cropped, cv.COLOR_BGR2GRAY)), cv.equalizeHist(pattern))
  pattern_resized = cv.resize(pattern, (img_matched.shape[1], img_matched.shape[0]))
  pattern_resized_rotate = rotate_finder(img_matched,pattern_resized)
  images, patterns, labels = window_sliding(img_matched, pattern_resized_rotate, img_name, crop_indexes,Kernel_size,Kernel_size)
  print(img_name, 'done!')
  return images,patterns,labels

In [12]:
images, patterns, labels = [], [], []
j=0
Kernel_size = 200
for img in os.listdir('Dataset'):
    if (img.endswith(".jpg")) | (img.endswith(".png")):
        i,p,l = pre_processing('Dataset/' + img)
        images.extend(i)
        patterns.extend(p)
        labels.extend(l)
        j += 1
    if j==100:
        break

Dataset/1644448758.2149394.png done!
Dataset/1644442073.9465039.png done!
Dataset/1644449241.8868785.png done!
Dataset/1644426961.8556552.png done!
Dataset/1644428265.7917786.png done!
Dataset/1644427828.5770006.png done!
Dataset/1644371311.8005233.jpg done!
Dataset/1644445546.5142424.png done!
Dataset/1644451795.3529818.png done!
Dataset/1644385988.2358375.jpg done!
Dataset/1644435639.6224587.png done!
Dataset/1644448357.0968075.png done!
Dataset/1644427076.3319407.png done!
Dataset/1644422935.2311373.png done!
Dataset/1644387674.0154285.jpg done!
Dataset/1644371982.7835793.jpg done!
Dataset/1644437322.3866885.png done!
Dataset/1644439245.3811543.png done!
Dataset/1644399350.3674173.jpg done!
Dataset/1644369801.9909956.jpg done!
Dataset/1644436152.2717178.png done!
Dataset/1644389271.051818.jpg done!
Dataset/1644441464.4244668.png done!
Dataset/1644380448.16593.jpg done!
Dataset/1644434445.597476.png done!
Dataset/1644451265.78203.png done!
Dataset/1644439331.6699107.png done!
Dataset

In [13]:
images = np.expand_dims(np.array(images).astype(np.float32), axis=3)
patterns= np.expand_dims(np.array(patterns).astype(np.float32), axis=3)
labels = np.expand_dims(np.array(labels).astype(np.float32), axis=1)

In [14]:
len(patterns)

14424

In [15]:
labels[labels==0].shape[0]/labels[labels==1].shape[0]

58.603305785123965

In [16]:
from keras import backend as K

# Model

In [17]:
def recall_m(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    recall = true_positives / (possible_positives + 
    K.epsilon())
    return recall

def precision_m(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    return precision

def f1_score_m(y_true, y_pred):
    precision = precision_m(y_true, y_pred)
    recall = recall_m(y_true, y_pred)
    return 2*((precision*recall)/(precision+recall+K.epsilon()))

In [18]:
# Provided two tensors t1 and t2
# Euclidean distance = sqrt(sum(square(t1-t2)))
def euclidean_distance(vects):
    """Find the Euclidean distance between two vectors.

    Arguments:
        vects: List containing two tensors of same length.

    Returns:
        Tensor containing euclidean distance
        (as floating point value) between vectors.
    """

    x, y = vects
    sum_square = tf.math.reduce_sum(tf.math.square(x - y), axis=1, keepdims=True)
    return tf.math.sqrt(tf.math.maximum(sum_square, tf.keras.backend.epsilon()))


input = layers.Input((120, 120, 1))
x = tf.keras.layers.BatchNormalization()(input)
x = layers.Conv2D(4, (5, 5), activation="tanh")(x)
x = layers.AveragePooling2D(pool_size=(2, 2))(x)
x = layers.Conv2D(16, (5, 5), activation="tanh")(x)
x = layers.AveragePooling2D(pool_size=(2, 2))(x)
x = layers.Flatten()(x)

x = tf.keras.layers.BatchNormalization()(x)
x = layers.Dense(10, activation="tanh")(x)
embedding_network = keras.Model(input, x)


input_1 = layers.Input((120, 120, 1))
input_2 = layers.Input((120, 120, 1))

# As mentioned above, Siamese Network share weights between
# tower networks (sister networks). To allow this, we will use
# same embedding network for both tower networks.
tower_1 = embedding_network(input_1)
tower_2 = embedding_network(input_2)

merge_layer = layers.Lambda(euclidean_distance)([tower_1, tower_2])
normal_layer = tf.keras.layers.BatchNormalization()(merge_layer)
output_layer = layers.Dense(1, activation="sigmoid")(normal_layer)
siamese = keras.Model(inputs=[input_1, input_2], outputs=output_layer)


In [19]:
def loss(margin=1):
    """Provides 'constrastive_loss' an enclosing scope with variable 'margin'.

    Arguments:
        margin: Integer, defines the baseline for distance for which pairs
                should be classified as dissimilar. - (default is 1).

    Returns:
        'constrastive_loss' function with data ('margin') attached.
    """

    # Contrastive loss = mean( (1-true_value) * square(prediction) +
    #                         true_value * square( max(margin-prediction, 0) ))
    def contrastive_loss(y_true, y_pred):
        """Calculates the constrastive loss.

        Arguments:
            y_true: List of labels, each label is of type float32.
            y_pred: List of predictions of same length as of y_true,
                    each label is of type float32.

        Returns:
            A tensor containing constrastive loss as floating point value.
        """

        square_pred = tf.math.square(y_pred)
        margin_square = tf.math.square(tf.math.maximum(margin - (y_pred), 0))
        return tf.math.reduce_mean(
            (1 - y_true) * square_pred + (y_true) * margin_square
        )

    return contrastive_loss


In [20]:
siamese.compile(loss=loss(margin=1), optimizer="RMSprop", metrics=['accuracy',recall_m,precision_m,f1_score_m])
siamese.summary()

Model: "model_1"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_2 (InputLayer)            [(None, 120, 120, 1) 0                                            
__________________________________________________________________________________________________
input_3 (InputLayer)            [(None, 120, 120, 1) 0                                            
__________________________________________________________________________________________________
model (Functional)              (None, 10)           165030      input_2[0][0]                    
                                                                 input_3[0][0]                    
__________________________________________________________________________________________________
lambda (Lambda)                 (None, 1)            0           model[0][0]                

In [21]:
class_weight = {0: 1, 1: (10*labels[labels==0].shape[0]/labels[labels==1].shape[0])}

history = siamese.fit(
    [images, patterns],
    labels,
    # validation_data=([x_val_1, x_val_2], labels_val),
    batch_size=1024,
    epochs=500,
    validation_split=0.2,
    callbacks=keras.callbacks.EarlyStopping(monitor='val_f1_score_m',
                                            patience=5,
                                            restore_best_weights=True),
    class_weight =class_weight
)

Epoch 1/500
12/12 [==============================] - 35s 3s/step - loss: 2.5640 - accuracy: 0.5270 - recall_m: 0.3664 - precision_m: 0.0126 - f1_score_m: 0.0244 - val_loss: 0.2481 - val_accuracy: 0.8707 - val_recall_m: 0.0000e+00 - val_precision_m: 0.0000e+00 - val_f1_score_m: 0.0000e+00
Epoch 2/500
12/12 [==============================] - 34s 3s/step - loss: 2.5210 - accuracy: 0.5563 - recall_m: 0.4729 - precision_m: 0.0176 - f1_score_m: 0.0339 - val_loss: 0.2558 - val_accuracy: 0.7005 - val_recall_m: 0.0145 - val_precision_m: 0.0011 - val_f1_score_m: 0.0021
Epoch 3/500
12/12 [==============================] - 34s 3s/step - loss: 2.4866 - accuracy: 0.6046 - recall_m: 0.4608 - precision_m: 0.0184 - f1_score_m: 0.0353 - val_loss: 0.2595 - val_accuracy: 0.5764 - val_recall_m: 0.0990 - val_precision_m: 0.0052 - val_f1_score_m: 0.0098
Epoch 4/500
12/12 [==============================] - 34s 3s/step - loss: 2.4551 - accuracy: 0.6425 - recall_m: 0.4837 - precision_m: 0.0210 - f1_score_m: 0.0

In [32]:
predictions

array([[0.49588948],
       [0.4885629 ],
       [0.48853737],
       ...,
       [0.4884933 ],
       [0.4884933 ],
       [0.4885743 ]], dtype=float32)

In [42]:
predictions = siamese.predict([images[0:1], patterns[0:1]])

In [46]:
newImage = []
newPattern = []
newLabels = []
for counter in range(len(predictions)):
    if predictions[counter] > 0.1:
        print(labels[counter])
        
    

[0.]


In [ ]:
def plt_metric(history, metric, title, has_valid=False):
    """Plots the given 'metric' from 'history'.

    Arguments:
        history: history attribute of History object returned from Model.fit.
        metric: Metric to plot, a string value present as key in 'history'.
        title: A string to be used as title of plot.
        has_valid: Boolean, true if valid data was passed to Model.fit else false.

    Returns:
        None.
    """
    plt.plot(history[metric])
    if has_valid:
        plt.plot(history["val_" + metric])
        plt.legend(["train", "validation"], loc="upper left")
    plt.title(title)
    plt.ylabel(metric)
    plt.xlabel("epoch")
    plt.show()


# Plot the accuracy
plt_metric(history=history.history, metric="accuracy", title="Model accuracy")

# Plot the constrastive loss
plt_metric(history=history.history, metric="loss", title="Constrastive Loss")

In [ ]:
predictions = siamese.predict([x_test_1, x_test_2])
visualize(pairs_test, labels_test, to_show=3, predictions=predictions, test=True)